# Gemma3-SD: Replace CLIP with Gemma 3 270M in Stable Diffusion 1.5

**Path B — UNet cross-attention dimension replacement (768 → 640)**

This notebook:
1. Trains a linear probe mapping Gemma 3 270M hidden states → CLIP hidden space
2. Surgically replaces all UNet cross-attention K/V layers from dim 768 → 640
3. Initializes new K/V weights using the trained probe (warm start)
4. LoRA fine-tunes the cross-attention layers on image-caption pairs
5. Runs inference with the Gemma-conditioned SD

**Requirements:** Colab T4 GPU (16GB VRAM), HuggingFace token (Gemma 3 is gated), wandb account (for logging)

> ⚠️ This is experimental. Changing UNet architecture destroys pretrained cross-attention weights.
> The probe warm-start helps but training is required for coherent images.

## Section 1: Environment Setup

In [ ]:
# @title 1.1 Install Dependencies (~3-5 min)
!pip install -q torch==2.1.0 torchvision --index-url https://download.pytorch.org/whl/cu118
!pip install -q transformers==4.46.0 accelerate==0.33.0 peft==0.12.0
!pip install -q diffusers==0.30.0 safetensors==0.4.3
!pip install -q datasets==2.20.0 xformers --index-url https://download.pytorch.org/whl/cu118
!pip install -q bitsandbytes==0.43.1
!pip install -q wandb matplotlib Pillow tqdm

import torch
import torch.nn as nn
print(f"PyTorch: {torch.__version__}, CUDA: {torch.version.cuda}, GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
# @title 1.2 Clone kohya-ss/sd-scripts (reference)
!git clone --depth 1 https://github.com/kohya-ss/sd-scripts.git /content/sd-scripts 2>/dev/null || echo "Already cloned"
import sys
sys.path.insert(0, '/content/sd-scripts')

In [ ]:
# @title 1.3 HuggingFace Login (Gemma 3 is gated)
from huggingface_hub import notebook_login
print("=" * 60)
print("⚠️  Gemma 3 270M is gated. You MUST:")
print("  1. Go to https://huggingface.co/google/gemma-3-270m-it")
print("  2. Accept the license agreement")
print("  3. Use a token with read access")
print("=" * 60)
notebook_login()

In [ ]:
# @title 1.4 Setup wandb Logging
import wandb
import os
from datetime import datetime

# Login to wandb (you'll need your API key from https://wandb.ai/authorize)
# Or set WANDB_API_KEY as a Colab secret
wandb.login()

run_name = f"gemma3-sd-{datetime.now().strftime('%Y%m%d-%H%M%S')}"
wandb.init(
    project="gemma3-stable-diffusion",
    name=run_name,
    config={
        "model": "Gemma 3 270M → SD 1.5 (Path B)",
        "gemma_hidden_size": 640,
        "original_cross_attn_dim": 768,
        "new_cross_attn_dim": 640,
        "lora_rank": 8,
        "lora_alpha": 16,
        "learning_rate": 1e-4,
        "probe_lr": 1e-3,
        "batch_size": 1,
        "max_prompt_length": 77,
        "device": str(torch.cuda.get_device_name(0)),
        "vram_gb": torch.cuda.get_device_properties(0).total_mem / 1e9,
    }
)
print(f"wandb run: {wandb.run.name}")

## Section 2: Train Linear Probe (Gemma 3 → CLIP Hidden Space)

Train Linear(640, 768) mapping Gemma hidden → CLIP hidden for cross-attention warm-start.

In [ ]:
# @title 2.1 Load Text Encoders
from transformers import AutoTokenizer, AutoModelForCausalLM, CLIPTextModel, CLIPTokenizer

device = torch.device("cuda")

# CLIP (SD 1.5 original)
print("Loading CLIP ViT-L/14...")
clip_tokenizer = CLIPTokenizer.from_pretrained("openai/clip-vit-large-patch14")
clip_model = CLIPTextModel.from_pretrained("openai/clip-vit-large-patch14").to(device).eval()
clip_hidden_size = clip_model.config.hidden_size
print(f"  CLIP hidden_size: {clip_hidden_size}")

# Gemma 3 270M
print("Loading Gemma 3 270M...")
gemma_tokenizer = AutoTokenizer.from_pretrained("google/gemma-3-270m-it")
gemma_model = AutoModelForCausalLM.from_pretrained(
    "google/gemma-3-270m-it", torch_dtype=torch.float16, device_map="auto"
).eval()
gemma_hidden_size = gemma_model.config.hidden_size
print(f"  Gemma hidden_size: {gemma_hidden_size}")
print(f"  Ratio CLIP/Gemma: {clip_hidden_size/gemma_hidden_size:.2f}")
wandb.log({"clip_hidden_size": clip_hidden_size, "gemma_hidden_size": gemma_hidden_size})

In [ ]:
# @title 2.2 Prepare Caption Dataset for Probe
import random
random.seed(42)

CAPTIONS = [
    "a red apple on a wooden table",
    "a cat sitting on a windowsill looking outside",
    "a futuristic cityscape at sunset with flying cars",
    "a watercolor painting of a mountain lake",
    "a macro photograph of a bee on a flower",
    "a vintage car parked on a cobblestone street",
    "an astronaut riding a horse on mars",
    "a cozy cabin in the snow with smoke coming from the chimney",
    "a neon-lit cyberpunk alleyway at night",
    "a field of sunflowers under a blue sky",
    "a detailed oil painting portrait of an old man",
    "a sushi platter on a black slate board",
    "a dragon flying over a medieval castle",
    "a cup of coffee with latte art on a wooden table",
    "a golden retriever puppy playing in the grass",
    "an abstract geometric pattern in pastel colors",
    "a steampunk airship floating above clouds",
    "a bowl of ramen with chopsticks and steam rising",
    "a forest path with sunlight filtering through trees",
    "a minimalist modern living room with large windows",
]

augmented = []
for cap in CAPTIONS:
    augmented.append(cap)
    augmented.append(cap + ", high quality, detailed")
print(f"Total captions: {len(augmented)}")
wandb.log({"probe_captions": len(augmented)})

In [ ]:
# @title 2.3 Collect Paired Embeddings
@torch.no_grad()
def get_clip_hidden(captions, batch_size=8):
    all_hidden = []
    for i in range(0, len(captions), batch_size):
        batch = captions[i:i+batch_size]
        tokens = clip_tokenizer(batch, return_tensors="pt", padding="max_length",
                               truncation=True, max_length=77).to(device)
        outputs = clip_model(**tokens)
        all_hidden.append(outputs.last_hidden_state.cpu())
    return torch.cat(all_hidden, dim=0)

@torch.no_grad()
def get_gemma_hidden(captions, batch_size=4, max_length=128):
    all_hidden = []
    for i in range(0, len(captions), batch_size):
        batch = captions[i:i+batch_size]
        tokens = gemma_tokenizer(batch, return_tensors="pt", padding="max_length",
                                truncation=True, max_length=max_length).to(device)
        outputs = gemma_model(**tokens, output_hidden_states=True)
        all_hidden.append(outputs.hidden_states[-1].cpu())
    return torch.cat(all_hidden, dim=0)

print("Collecting CLIP embeddings...")
clip_hidden = get_clip_hidden(augmented)  # [N, 77, 768]
print(f"  CLIP: {clip_hidden.shape}")

print("Collecting Gemma embeddings...")
gemma_hidden = get_gemma_hidden(augmented)  # [N, S, 640]
print(f"  Gemma: {gemma_hidden.shape}")

min_seq = min(clip_hidden.shape[1], gemma_hidden.shape[1])
clip_aligned = clip_hidden[:, :min_seq, :]
gemma_aligned = gemma_hidden[:, :min_seq, :]
print(f"  Aligned: CLIP {clip_aligned.shape}, Gemma {gemma_aligned.shape}")

In [ ]:
# @title 2.4 Train the Probe
class Probe(nn.Module):
    def __init__(self, gemma_dim, clip_dim):
        super().__init__()
        self.linear = nn.Linear(gemma_dim, clip_dim)
    def forward(self, x):
        return self.linear(x)

probe = Probe(gemma_hidden_size, clip_hidden_size).to(device)
opt = torch.optim.AdamW(probe.parameters(), lr=1e-3)
criterion = nn.MSELoss()

gemma_flat = gemma_aligned.reshape(-1, gemma_hidden_size).to(device)
clip_flat = clip_aligned.reshape(-1, clip_hidden_size).to(device)
print(f"Training data: {gemma_flat.shape[0]} token pairs")
print(f"Params: {sum(p.numel() for p in probe.parameters()):,}")

probe.train()
batch_size = 1024
for epoch in range(500):
    perm = torch.randperm(gemma_flat.shape[0])
    total = 0.0
    for i in range(0, gemma_flat.shape[0], batch_size):
        idx = perm[i:i+batch_size]
        pred = probe(gemma_flat[idx])
        loss = criterion(pred, clip_flat[idx])
        opt.zero_grad()
        loss.backward()
        opt.step()
        total += loss.item()
    if epoch % 50 == 0:
        print(f"  Epoch {epoch:3d}: loss = {total / (gemma_flat.shape[0] // batch_size):.6f}")

# Cosine similarity check
with torch.no_grad():
    sim = nn.functional.cosine_similarity(probe(gemma_flat[:100]), clip_flat[:100], dim=1)
    print(f"Mean cosine sim: {sim.mean():.4f}")

torch.save({k: v.cpu() for k, v in probe.state_dict().items()}, "/content/probe.pt")
wandb.log({"probe_cosine_sim": sim.mean().item()})
print("Probe saved!")

## Section 3: UNet Surgery

Replace all cross-attention to_k/to_v layers from dim 768→640. Initialize new weights from probe: W_new = W_old @ probe

In [ ]:
# @title 3.1 Load SD 1.5
from diffusers import StableDiffusionPipeline
import gc

print("Loading SD 1.5...")
pipe = StableDiffusionPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5", torch_dtype=torch.float16, safety_checker=None
)
unet = pipe.unet.to(device).eval()
vae = pipe.vae.to(device).eval()
del pipe; gc.collect(); torch.cuda.empty_cache()

print(f"UNet cross_attention_dim: {unet.config.cross_attention_dim}")
print(f"UNet params: {sum(p.numel() for p in unet.parameters()) / 1e6:.1f}M")
wandb.log({"unet_params_M": sum(p.numel() for p in unet.parameters()) / 1e6})

In [ ]:
# @title 3.2 Identify & Replace Cross-Attention Layers
probe_state = torch.load("/content/probe.pt", map_location=device)
probe_weight = probe_state['linear.weight']  # [768, 640]
probe_bias = probe_state.get('linear.bias', None)
new_dim = gemma_hidden_size  # 640

print(f"Probe weight: {probe_weight.shape}")

replacements = 0
for name, module in unet.named_modules():
    if hasattr(module, 'to_k') and module.to_k.in_features == 768:
        inner_dim = module.to_k.out_features
        has_bias = module.to_k.bias is not None

        # W_new = W_old @ probe_weight
        with torch.no_grad():
            new_k_w = module.to_k.weight.data @ probe_weight
            new_v_w = module.to_v.weight.data @ probe_weight

        new_k = nn.Linear(new_dim, inner_dim, bias=has_bias).to(device)
        new_v = nn.Linear(new_dim, inner_dim, bias=has_bias).to(device)
        new_k.weight.data.copy_(new_k_w)
        new_v.weight.data.copy_(new_v_w)
        if has_bias:
            new_k.bias.data.copy_(module.to_k.bias.data)
            new_v.bias.data.copy_(module.to_v.bias.data)

        module.to_k = new_k
        module.to_v = new_v
        replacements += 1

unet.config.cross_attention_dim = new_dim
print(f"Replaced {replacements} cross-attention blocks")

# Verify
matching = sum(1 for _, m in unet.named_modules()
               if hasattr(m, 'to_k') and m.to_k.in_features == new_dim)
leftover = sum(1 for _, m in unet.named_modules()
               if hasattr(m, 'to_k') and m.to_k.in_features == 768)
print(f"New-dim layers: {matching}, Leftover 768: {leftover}")
assert leftover == 0, f"{leftover} layers still at 768!"
print("✓ All cross-attention dimensions replaced!")
wandb.log({"cross_attn_blocks_replaced": replacements, "new_cross_dim": new_dim})

In [ ]:
# @title 3.3 Verify Forward Pass
@torch.no_grad()
def test_forward(prompt="a cat on a table"):
    tokens = gemma_tokenizer(prompt, return_tensors="pt", padding="max_length",
                            truncation=True, max_length=77).to(device)
    out = gemma_model(**tokens, output_hidden_states=True)
    ehs = out.hidden_states[-1]  # [1, 77, 640]

    latents = torch.randn(1, 4, 64, 64, device=device, dtype=torch.float16)
    t = torch.tensor([500], device=device)
    result = unet(latents, t, encoder_hidden_states=ehs).sample
    print(f"Input: {ehs.shape}, Output: {result.shape}")
    return result

test_forward()
print("✓ Forward pass OK!")

## Section 4: LoRA Training on Cross-Attention

Freeze VAE + all UNet except LoRA on attn2.to_k/attn2.to_v

## Section 4A: Dataset Setup

### Format
Each image needs a paired caption file:
```
training_data/
├── image_001.png
├── image_001.txt      ← "a cat on a windowsill"
├── image_002.jpg
├── image_002.txt      ← "sunset over mountains"
└── ...
```
Supported image formats: .png, .jpg, .jpeg. Caption files: .txt or .caption.

### Methods to Provide Data
- **Google Drive** (recommended): zip your dataset, upload to Drive, mount below
- **Direct upload**: drag folder into Colab's file panel → `/content/training_data/`
- **HuggingFace Hub**: `datasets.load_dataset("your/dataset")`

In [ ]:
# @title 4A.1 Load Dataset (choose method)import os, zipfilefrom google.colab import driveDATASET_DIR = "/content/training_data"os.makedirs(DATASET_DIR, exist_ok=True)# === METHOD 1: Google Drive (recommended) ===# 1. Upload your_dataset.zip to Google Drive root# 2. Uncomment below:# drive.mount('/content/drive')# !cp /content/drive/MyDrive/your_dataset.zip /content/# with zipfile.ZipFile('/content/your_dataset.zip', 'r') as z:#     z.extractall(DATASET_DIR)# === METHOD 2: HuggingFace Hub ===# Option A: Use download script (max 1GB zip)# !wget -O download_hf_dataset.py https://raw.githubusercontent.com/...  # or copy into Colab# !python download_hf_dataset.py --output /content/dataset.zip --max-size-mb 950# !unzip -q /content/dataset.zip -d /content/training_data/## Option B: Direct load (not size-limited)# from datasets import load_dataset# ds = load_dataset("your/dataset", split="train")# for i, sample in enumerate(ds):#     sample["image"].save(f"{DATASET_DIR}/img_{i:05d}.png")#     with open(f"{DATASET_DIR}/img_{i:05d}.txt", "w") as f:#         f.write(sample["text"])# === METHOD 3: Direct upload ===# Already done if you dragged files to /content/training_data/# === Validate dataset ===images = [f for f in os.listdir(DATASET_DIR) if f.endswith(('.png','.jpg','.jpeg'))]captions = [f for f in os.listdir(DATASET_DIR) if f.endswith(('.txt','.caption'))]print(f"Found {len(images)} images, {len(captions)} captions")if len(images) == 0:    print("\n⚠️  NO IMAGES FOUND! Creating demo noise dataset instead...")    print("    Results will be random noise. Provide real data for training.")    import numpy as np    from PIL import Image    DEMO_CAPS = ["a landscape", "a cat", "a building", "pasta", "a car",                 "a beach", "flowers", "a room", "a robot", "fruit",                 "a dog", "a garden", "coffee", "snow", "neon",                 "a forest", "a bicycle", "ramen", "a cat at window", "sunset"]    for i in range(len(DEMO_CAPS)):        img = Image.fromarray(np.random.randint(0, 255, (512, 512, 3), dtype=np.uint8))        img.save(f"{DATASET_DIR}/img_{i:04d}.png")        with open(f"{DATASET_DIR}/img_{i:04d}.txt", "w") as f:            f.write(DEMO_CAPS[i])    images = [f for f in os.listdir(DATASET_DIR) if f.endswith(('.png','.jpg','.jpeg'))]    print(f"Created {len(images)} demo images.")assert len(images) > 0, "No images loaded!"print(f"Dataset ready: {len(images)} images")

In [ ]:
# @title 4.1 Manual LoRA for Cross-Attention Only
class ManualLoRA(nn.Module):
    """LoRA wrapper for a single Linear layer."""
    def __init__(self, base_linear, rank=8, alpha=16):
        super().__init__()
        self.base = base_linear
        self.rank = rank
        self.scaling = alpha / rank
        in_f, out_f = base_linear.in_features, base_linear.out_features
        self.lora_A = nn.Linear(in_f, rank, bias=False)
        self.lora_B = nn.Linear(rank, out_f, bias=False)
        nn.init.kaiming_uniform_(self.lora_A.weight, a=5**0.5)
        nn.init.zeros_(self.lora_B.weight)
        for p in base_linear.parameters():
            p.requires_grad = False

    def forward(self, x):
        return self.base(x) + self.lora_B(self.lora_A(x)) * self.scaling

# Freeze everything first
for p in unet.parameters():
    p.requires_grad = False

# Apply LoRA only to attn2 to_k and to_v
lora_count = 0
for name, module in unet.named_modules():
    if hasattr(module, 'to_k') and 'attn2' in name and module.to_k.in_features == new_dim:
        module.to_k = ManualLoRA(module.to_k, rank=8, alpha=16)
        module.to_v = ManualLoRA(module.to_v, rank=8, alpha=16)
        lora_count += 2

trainable = sum(p.numel() for p in unet.parameters() if p.requires_grad)
total = sum(p.numel() for p in unet.parameters())
print(f"LoRA layers: {lora_count}")
print(f"Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")
wandb.log({"lora_layers": lora_count, "trainable_params": trainable, "total_params": total})

In [ ]:
# @title 4.2 Define Dataset Class (with bucketing)
import os, math
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

# Bucket resolutions — SD 1.5 native, no upscale, max 512px on short side
BUCKETS = [
    (512, 512),   # square
    (512, 768),   # portrait 2:3
    (768, 512),   # landscape 3:2
    (448, 704),   # portrait 5:8
    (704, 448),   # landscape 8:5
    (384, 640),   # portrait 3:5
    (640, 384),   # landscape 5:3
]

def get_bucket(w, h):
    """Return (bw, bh) of nearest aspect-ratio bucket."""
    target = w / h
    best = None
    best_dist = float('inf')
    for bw, bh in BUCKETS:
        dist = abs(bw / bh - target)
        if dist < best_dist:
            best_dist = dist
            best = (bw, bh)
    return best

class SDDataset(Dataset):
    """Image-caption pairs with aspect-ratio bucketing."""
    def __init__(self, image_dir, tokenizer, max_length=77):
        self.dir = image_dir
        self.tok = tokenizer
        self.ml = max_length
        self.imgs = sorted([
            f for f in os.listdir(image_dir)
            if f.lower().endswith(('.png', '.jpg', '.jpeg'))
        ])
        self.caps = []
        self.buckets = []  # (bw, bh) per image
        for f in self.imgs:
            # Load caption
            base = os.path.splitext(f)[0]
            cap = ""
            for ext in ['.txt', '.caption']:
                p = os.path.join(image_dir, base + ext)
                if os.path.exists(p):
                    with open(p) as fp:
                        cap = fp.read().strip()
                    break
            self.caps.append(cap)
            # Determine bucket
            img_path = os.path.join(image_dir, f)
            with Image.open(img_path) as im:
                bw, bh = get_bucket(im.width, im.height)
            self.buckets.append((bw, bh))
        # Count per bucket
        from collections import Counter
        bc = Counter(self.buckets)
        print(f"Buckets: {dict(sorted(bc.items()))}")

    def __len__(self):
        return len(self.imgs)

    def __getitem__(self, i):
        img_path = os.path.join(self.dir, self.imgs[i])
        img = Image.open(img_path).convert("RGB")
        bw, bh = self.buckets[i]
        # Resize to bucket (no crop, preserves aspect ratio)
        img = img.resize((bw, bh), Image.LANCZOS)
        # Normalize to [-1, 1]
        img_tensor = transforms.ToTensor()(img) * 2 - 1
        tokens = self.tok(
            self.caps[i], return_tensors="pt",
            padding="max_length", truncation=True, max_length=self.ml
        )
        return {
            "image": img_tensor,
            "bucket": (bw, bh),
            "input_ids": tokens.input_ids[0],
            "attention_mask": tokens.attention_mask[0],
        }

ds = SDDataset("/content/training_data", gemma_tokenizer)
dl = DataLoader(ds, batch_size=1, shuffle=True)
print(f"Dataset: {len(ds)} images, {len(BUCKETS)} bucket sizes")

In [ ]:
# @title 4.3 Precompute VAE Latents (variable sizes)
from diffusers import AutoencoderKL

cache = {}
vae.eval()
with torch.no_grad():
    for i in range(len(ds)):
        batch = ds[i]
        img = batch["image"].unsqueeze(0).to(device, dtype=torch.float16)
        latent = vae.encode(img).latent_dist.sample() * vae.config.scaling_factor
        cache[i] = {
            "latent": latent.cpu(),
            "input_ids": batch["input_ids"],
            "attention_mask": batch["attention_mask"],
        }
    if i % 50 == 0:
        print(f"  {i+1}/{len(ds)} latents cached")
print(f"Cached {len(cache)} latents (variable shapes per bucket)")

vae = vae.cpu()
import gc; gc.collect(); torch.cuda.empty_cache()

In [ ]:
# @title 4.4 Training Loopfrom diffusers import DDPMSchedulerfrom tqdm import tqdmscheduler = DDPMScheduler.from_pretrained("runwayml/stable-diffusion-v1-5", subfolder="scheduler")optimizer = torch.optim.AdamW([p for p in unet.parameters() if p.requires_grad], lr=1e-4, weight_decay=0.01)unet.train()num_epochs = 10global_step = 0print(f"Training {num_epochs} epochs, {len(ds)} images")print("  (batch_size=1 required for variable bucket sizes)")wandb.watch(unet, log="gradients", log_freq=50)for epoch in range(num_epochs):    epoch_loss = 0.0    indices = list(range(len(ds)))    np.random.shuffle(indices)    progress = tqdm(indices, desc=f"Epoch {epoch+1}/{num_epochs}")    for i in progress:        c = cache[i]        latent = c["latent"].to(device, dtype=torch.float16)        input_ids = c["input_ids"].unsqueeze(0).to(device)        attn_mask = c["attention_mask"].unsqueeze(0).to(device)        with torch.no_grad():            gemma_out = gemma_model(input_ids=input_ids, attention_mask=attn_mask, output_hidden_states=True)            ehs = gemma_out.hidden_states[-1].to(dtype=torch.float16)        noise = torch.randn_like(latent)        t = torch.randint(0, scheduler.config.num_train_timesteps, (1,), device=device).long()        noisy = scheduler.add_noise(latent, noise, t)        pred = unet(noisy, t, encoder_hidden_states=ehs).sample        loss = nn.functional.mse_loss(pred.float(), noise.float())        optimizer.zero_grad()        loss.backward()        nn.utils.clip_grad_norm_([p for p in unet.parameters() if p.requires_grad], 1.0)        optimizer.step()        epoch_loss += loss.item()        global_step += 1        progress.set_postfix({"loss": f"{loss.item():.4f}"})        if global_step % 10 == 0:            wandb.log({"train/loss": loss.item(), "train/step": global_step})    avg_loss = epoch_loss / len(indices)    print(f"Epoch {epoch+1}: avg_loss = {avg_loss:.4f}")    wandb.log({"train/epoch_loss": avg_loss, "train/epoch": epoch})    if (epoch + 1) % 5 == 0:        lora_w = {}        for n, p in unet.named_parameters():            if p.requires_grad: lora_w[n] = p.data.cpu().clone()        torch.save(lora_w, f"/content/lora_epoch{epoch+1}.pt")        print(f"  Checkpoint saved")# Save finallora_w = {}for n, p in unet.named_parameters():    if p.requires_grad: lora_w[n] = p.data.cpu().clone()torch.save(lora_w, "/content/gemma3_sd_lora.pt")print("Training complete! LoRA saved.")wandb.finish()

## Section 5: Inference

Generate images with the Gemma-conditioned SD.

In [ ]:
# @title 5.1 Generate Image
import matplotlib.pyplot as plt
from diffusers import DDPMScheduler

vae = vae.to(device).eval()
unet.eval()
scheduler = DDPMScheduler.from_pretrained("runwayml/stable-diffusion-v1-5", subfolder="scheduler")
scheduler.set_timesteps(50)

@torch.no_grad()
def generate(prompt, steps=30, guidance=7.5, seed=42):
    gen = torch.Generator(device=device).manual_seed(seed)

    tok = gemma_tokenizer(prompt, return_tensors="pt", padding="max_length",
                          truncation=True, max_length=77).to(device)
    ehs = gemma_model(**tok, output_hidden_states=True).hidden_states[-1].to(torch.float16)

    uncond = gemma_tokenizer("", return_tensors="pt", padding="max_length",
                            truncation=True, max_length=77).to(device)
    uncond_h = gemma_model(**uncond, output_hidden_states=True).hidden_states[-1].to(torch.float16)
    ehs = torch.cat([uncond_h, ehs])

    latents = torch.randn(1, 4, 64, 64, generator=gen, device=device, dtype=torch.float16)

    for t in tqdm(scheduler.timesteps[:steps], desc="Generating"):
        inp = torch.cat([latents] * 2)
        inp = scheduler.scale_model_input(inp, t)
        pred = unet(inp, t, encoder_hidden_states=ehs).sample
        u, p = pred.chunk(2)
        pred = u + guidance * (p - u)
        latents = scheduler.step(pred, t, latents).prev_sample

    latents = latents / vae.config.scaling_factor
    img = vae.decode(latents).sample
    img = (img/2 + 0.5).clamp(0,1).cpu().permute(0,2,3,1).float().numpy()
    return Image.fromarray((img[0]*255).astype(np.uint8))

prompts = [
    "a cat sitting on a windowsill looking outside",
    "a watercolor painting of a mountain lake",
    "a neon-lit cyberpunk alleyway at night",
]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, p in zip(axes, prompts):
    print(f"Generating: {p}")
    img = generate(p, steps=30)
    ax.imshow(img)
    ax.set_title(p[:40]+"...", fontsize=10)
    ax.axis('off')
plt.tight_layout()
plt.savefig("/content/samples.png", dpi=100)
plt.show()

In [ ]:
# @title 5.2 Save Complete Model
save = {
    "unet_state_dict": unet.state_dict(),
    "unet_config": unet.config.to_dict(),
    "probe_state_dict": torch.load("/content/probe.pt", map_location="cpu"),
    "lora_state_dict": lora_w,
    "gemma_hidden_size": gemma_hidden_size,
    "new_cross_attention_dim": new_dim,
}
torch.save(save, "/content/gemma3_sd_complete.pt")
print(f"Complete model saved ({os.path.getsize('/content/gemma3_sd_complete.pt')/1e9:.2f} GB)")

## Roadmap & Next Steps

### To Get Good Results
- **Replace demo images** with 500+ real image-caption pairs
- **Train longer**: 50-100 epochs with real data
- **Higher LoRA rank**: 16-32 for more expressiveness
- **Stage 2**: Add UNet self-attention LoRA once cross-attention converges
- **Longer prompts**: Increase max_length to 128-256 tokens

### wandb Dashboard
Check your wandb project for loss curves, gradient histograms, and generated samples.

### Architecture Notes
- Cross-attention is reinitialized as `W_new = W_old @ probe` preserving semantic mapping
- This is equivalent to `K_gemma(g) = K_clip(probe(g))` ≈ `K_clip(CLIP(prompt))`
- Training on image pairs teaches UNet to read Gemma's native representation
- Gemma 3 supports 32K tokens → future work: longer prompt conditioning

## Appendix: download_hf_dataset.py

Run this cell to write the download script to disk, then execute it above.

In [ ]:
# @title Write download script to disk
# Run this, then go back to cell 4A.1 and use Method 2 / Option A
script = r'''#!/usr/bin/env python3
"""
Streaming-zip downloader: RobinWZQ/improved_aesthetics_6.5plus → max-1GB zip.

Writes directly to the output zip as samples arrive. After each image+caption
pair is added, checks the on-disk zip size. Stops when the ZIP FILE exceeds the
limit (not the uncompressed data).

Ensures proper image-txt pairing: img_000000.jpg + img_000000.txt in flat layout.

Usage:
    python download_hf_dataset.py --output dataset.zip --max-size-mb 950

Colab:
    !python download_hf_dataset.py --output /content/dataset.zip
    !unzip -q /content/dataset.zip -d /content/training_data/
"""

import argparse
import io
import os
import sys
import zipfile
from pathlib import Path

parser = argparse.ArgumentParser(description="Streaming zip: HF dataset → max-1GB zip")
parser.add_argument("--output", default="dataset.zip", help="Output zip path")
parser.add_argument("--max-size-mb", type=int, default=950, help="Max ZIP FILE size in MB")
parser.add_argument("--dataset", default="RobinWZQ/improved_aesthetics_6.5plus")
parser.add_argument("--split", default="train")
parser.add_argument("--image-col", default="image", help="Column with PIL image")
parser.add_argument("--text-col", default="text", help="Column with caption")
parser.add_argument("--start", type=int, default=0, help="Skip first N samples")
args = parser.parse_args()

# ── Deps ─────────────────────────────────────────────────────────
try:
    from datasets import load_dataset
except ImportError:
    sys.exit("ERROR: pip install datasets pillow")
try:
    from PIL import Image
except ImportError:
    sys.exit("ERROR: pip install pillow")

# ── Load dataset (streaming) ─────────────────────────────────────
print(f"Streaming {args.dataset} [{args.split}] ...")
ds = load_dataset(args.dataset, split=args.split, streaming=True)

max_bytes = args.max_size_mb * 1024 * 1024
saved = 0
skipped_no_cap = 0
skipped_no_img = 0
errors = 0

# ── Open streaming zip ───────────────────────────────────────────
# We'll write to a temp file first so we can measure size.
# Python ZipFile doesn't support removing entries, so we write to
# a BytesIO buffer in memory and flush to disk only when done.
# BUT: for 1GB this is too much memory. Instead, we use a staging
# approach: write each pair to a temporary zip, check size, if under
# limit copy to main zip as we go. Actually simplest: write to a temp
# zip file, then rename at the end.
#
# BEST approach for streaming: use zipfile.ZipFile in write mode
# on the output file. After each writestr/write, close and reopen
# to check size. If over, delete the output and report where we stopped.

# Simpler: accumulate pairs in a list of (name, data_bytes) until
# projected zip size exceeds limit, then flush all at once.
# But we need to measure compressed size, not raw size.

# SIMPLEST that actually works: write to disk incrementally.
# Use a temporary directory for staging, but write pairs directly
# into a ZipFile. After each pair, check the zip's file size.
# If it crossed the limit, we need to remove the last entry.
# Since ZipFile can't remove, we'll write to a NEW temp zip each time
# and swap. Overhead: copying the zip N times. For 500 images × 1MB zip
# this is ~500MB of extra writes. Acceptable.

# ACTUALLY SIMPLEST: write to temp dir, then zip at the end.
# Track uncompressed size as a rough proxy. JPEGs don't compress
# further in zip, so uncompressed ≈ zip size. Add 5% margin.

import tempfile
import shutil

tmpdir = tempfile.mkdtemp(prefix="hf_stream_")
total_raw = 0
MAX_RAW = int(max_bytes * 0.90)  # 10% margin for zip overhead + txt files

print(f"Target: {args.max_size_mb} MB zip  →  ~{MAX_RAW / 1024**2:.0f} MB raw limit")
print(f"Temp staging: {tmpdir}")

ds_iter = iter(ds)
for _ in range(args.start):
    try:
        next(ds_iter)
    except StopIteration:
        break

for sample in ds_iter:
    # ── Image ────────────────────────────────────────────────────
    img = sample.get(args.image_col)
    if img is None:
        skipped_no_img += 1
        continue

    # Normalize to PIL
    if isinstance(img, Image.Image):
        pil_img = img
    elif isinstance(img, dict) and 'bytes' in img:
        pil_img = Image.open(io.BytesIO(img['bytes']))
    elif isinstance(img, bytes):
        pil_img = Image.open(io.BytesIO(img))
    else:
        skipped_no_img += 1
        continue

    # ── Caption ──────────────────────────────────────────────────
    caption = ""
    for key in [args.text_col, 'caption', 'prompt', 'text_en']:
        if key in sample and sample[key]:
            caption = str(sample[key]).strip()
            break
    if not caption:
        skipped_no_cap += 1
        continue

    # ── Encode image to JPEG bytes ───────────────────────────────
    if pil_img.mode in ('RGBA', 'P', 'LA'):
        pil_img = pil_img.convert('RGB')
    buf = io.BytesIO()
    try:
        pil_img.save(buf, format='JPEG', quality=92)
        img_bytes = buf.getvalue()
    except Exception as e:
        errors += 1
        continue

    caption_bytes = caption.encode('utf-8')

    # ── Check if adding this pair would exceed limit ─────────────
    pair_raw = len(img_bytes) + len(caption_bytes)
    if total_raw + pair_raw > MAX_RAW:
        print(f"  Limit reached at {saved} images ({total_raw/1024**2:.1f} MB raw)")
        break

    # ── Write to staging ─────────────────────────────────────────
    img_name = f"img_{saved:06d}.jpg"
    txt_name = f"img_{saved:06d}.txt"
    with open(os.path.join(tmpdir, img_name), 'wb') as f:
        f.write(img_bytes)
    with open(os.path.join(tmpdir, txt_name), 'w', encoding='utf-8') as f:
        f.write(caption)

    total_raw += pair_raw
    saved += 1

    if saved % 100 == 0:
        print(f"  {saved} images | {total_raw / 1024**2:.1f} MB raw")

# ── Zip it ──────────────────────────────────────────────────────
print(f"\n{saved} images (no_cap={skipped_no_cap}, no_img={skipped_no_img}, errors={errors})")
print(f"Raw total: {total_raw / 1024**2:.1f} MB")
print(f"Creating zip: {args.output} ...")

with zipfile.ZipFile(args.output, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fname in sorted(os.listdir(tmpdir)):
        zf.write(os.path.join(tmpdir, fname), fname)

zip_size = os.path.getsize(args.output)
print(f"Zip size: {zip_size / 1024**2:.1f} MB")

if zip_size > max_bytes:
    print(f"WARNING: zip exceeds {args.max_size_mb} MB limit!")
    print(f"  Overshoot: {(zip_size - max_bytes) / 1024**2:.1f} MB")
    print(f"  Re-run with --max-size-mb {int(zip_size/1024**2 * 0.85)} for safety")

# ── Validate pairing ────────────────────────────────────────────
print("\nValidating pair integrity...")
with zipfile.ZipFile(args.output, 'r') as zf:
    names = zf.namelist()
    imgs = sorted([n for n in names if n.endswith(('.jpg', '.jpeg', '.png'))])
    txts = sorted([n for n in names if n.endswith('.txt')])
    orphan_imgs = 0
    orphan_txts = 0
    for n in imgs:
        expected = os.path.splitext(n)[0] + '.txt'
        if expected not in txts:
            orphan_imgs += 1
    for n in txts:
        expected = os.path.splitext(n)[0] + '.jpg'
        if expected not in imgs:
            orphan_txts += 1
    print(f"  Images: {len(imgs)}, Captions: {len(txts)}")
    print(f"  Orphans: {orphan_imgs} images, {orphan_txts} captions")
    if orphan_imgs == 0 and orphan_txts == 0:
        print("  ✓ All pairs intact")

# ── Cleanup ─────────────────────────────────────────────────────
shutil.rmtree(tmpdir, ignore_errors=True)

output_abs = os.path.abspath(args.output)
print(f"\nDone → {output_abs}")
print(f"Colab usage:")
print(f"  !unzip -q {args.output} -d /content/training_data/")
'''
with open('/content/download_hf_dataset.py', 'w') as f:
    f.write(script)
print('Script written to /content/download_hf_dataset.py')
print('Now run: !python /content/download_hf_dataset.py --output /content/dataset.zip')
